# Data Exploration — NLP Sentiment Analyzer

This notebook explores the combined sentiment dataset: class distribution,
text length, word frequencies, and source-wise breakdown.

**Datasets**: Amazon Reviews, Twitter Sentiment140, Hotel Reviews

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

LABEL_NAMES = {0: 'negative', 1: 'neutral', 2: 'positive'}

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/combined_dataset.csv')
print(f'Total samples: {len(df):,}')
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df['label'].value_counts().sort_index()
axes[0].bar([LABEL_NAMES[i] for i in counts.index], counts.values, color=['#ef4444', '#eab308', '#22c55e'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

# Pie chart
axes[1].pie(counts.values, labels=[LABEL_NAMES[i] for i in counts.index], autopct='%1.1f%%', colors=['#ef4444', '#eab308', '#22c55e'])
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

## 3. Text Length Distribution

In [ ]:
df['text_length'] = df['clean_text'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, label in enumerate([0, 1, 2]):
    subset = df[df['label'] == label]
    axes[i].hist(subset['text_length'], bins=50, color=['#ef4444', '#eab308', '#22c55e'][i], alpha=0.7)
    axes[i].set_title(f'{LABEL_NAMES[label].capitalize()} (n={len(subset):,})')
    axes[i].set_xlabel('Text Length (chars)')
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()
print(f'\nMean text length: {df["text_length"].mean():.0f} chars')
print(f'Median text length: {df["text_length"].median():.0f} chars')

## 4. Word Clouds per Sentiment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = ['#ef4444', '#eab308', '#22c55e']

for i, label in enumerate([0, 1, 2]):
    text = ' '.join(df[df['label'] == label]['clean_text'].sample(min(5000, len(df[df['label'] == label])), random_state=42))
    wordcloud = WordCloud(width=800, height=400, background_color='black', colormap='viridis', max_words=100).generate(text)
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(LABEL_NAMES[label].capitalize(), fontsize=16, color=colors[i])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Source-wise Distribution

In [ ]:
if 'source' in df.columns:
    pivot = df.groupby(['source', 'label']).size().unstack(fill_value=0)
    pivot.columns = [LABEL_NAMES[c] for c in pivot.columns]
    pivot.plot(kind='bar', stacked=True, color=['#ef4444', '#eab308', '#22c55e'], figsize=(10, 6))
    plt.title('Sentiment Distribution by Source')
    plt.xlabel('Source')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.legend(title='Sentiment')
    plt.tight_layout()
    plt.show()
    print(pivot)